# **Hybrid Search**

### hybrid search is the combinasion between the vector search that we already saw with embeddings, and keyword search that is traditionally used to match exactly the text, with both having some weaknesses, the mix of them creates a very powerfully search

In [3]:
from langchain_community.retrievers import  BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq
load_dotenv()

c:\Users\Admin\Desktop\RAG-learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [ ]:
documents = [Document(page_content=chunk,metadata={"source":f"chunk {i+1}"}) for i,chunk in enumerate(chunks)]
documents[0]

Document(metadata={'source': 'chunk number 1'}, page_content='Microsoft acquired GitHub for 7.5 billion dollars in 2018.')

## **Retrivers**

### **1-vector retriever**

In [9]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
dir = "../db/hybrid_db"
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hsnw:space":"cosine"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3031.31it/s]


In [15]:
vector_retriever = vector_store.as_retriever(search_kwargs={"k":2})
test_vector_query = "space exploration company"
relevent_docs = vector_retriever.invoke(test_vector_query)
for doc in relevent_docs:
    print(doc.page_content)

SpaceX develops Starship rockets for Mars missions.
Google is a large technology company with global operations.


### **2-BM25 retriever**

In [16]:
bm25_retriver = BM25Retriever.from_documents(documents)
bm25_retriver.k = 2

In [18]:
test_keyword_query = "Tesla"
relevent_docs_keyword = bm25_retriver.invoke(test_keyword_query)
for rel in relevent_docs_keyword:
    print(rel.page_content)

Tesla Tesla Tesla financial quarterly results improved significantly.
Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.


## **3-hybrid search**

In [21]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriver,vector_retriever],
    weights=[0.5,0.5]
)

In [23]:
hybrid_test_query = "purchase cost 7.5 billion"

relevent_docs_hybrid = hybrid_retriever.invoke(hybrid_test_query)

for rel in relevent_docs_hybrid:
    print(rel.page_content)

Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Orange County reported new housing developments.
Microsoft is a large technology company with global operations.


## **LLM**

In [25]:
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
prompt = f"""answer the following question based on the documents provided, query : {hybrid_test_query}
    Documents : 
    {chr(10).join([doc.page_content for doc in relevent_docs_hybrid])}
    provide an answer only based on the documents, if you dont find the  answer in them, just so no enough informations for the question
"""
final_prompt = [{
    "role":"user",
    "content":prompt
}]
client = Groq(api_key=GROQ_API_KEY)
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0,
    messages=final_prompt
)
print(response.choices[0].message.content)

Microsoft acquired GitHub for a purchase cost of 7.5 billion dollars in 2018.
